In [4]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("PairRDDs")
  .master("local[*]")
  .getOrCreate()

val sc = spark.sparkContext

// RDD de ventas como cadenas: "producto,región,importe"
val ventas = sc.parallelize(List(
  "Laptop,Norte,1200",
  "Teclado,Sur,45",
  "Monitor,Norte,350",
  "Laptop,Sur,1200",
  "Ratón,Norte,25",
  "Monitor,Sur,350",
  "Teclado,Norte,45"
))

// Convertimos a Pair RDD: (región, importe)
val ventasPorRegion = ventas.map { linea =>
  val campos = linea.split(",")
  val region  = campos(1)
  val importe = campos(2).toDouble
  (region, importe)   // ← tupla (clave, valor)
}

ventasPorRegion.collect().foreach(println)
// (Norte,1200.0)
// (Sur,45.0)
// (Norte,350.0)
// (Sur,1200.0)
// (Norte,25.0)
// (Sur,350.0)
// (Norte,45.0)

// Objetivo: para cada región → List con todos los importes
val importesPorRegion = ventasPorRegion.combineByKey(
  (v: Double) => List(v),                       // createCombiner: primer valor → List
  (acc: List[Double], v: Double) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

importesPorRegion.collect().foreach { case (region, lista) =>
  println(s"$region → $lista")
}
// Norte → List(1200.0, 350.0, 25.0, 45.0)
// Sur   → List(45.0, 1200.0, 350.0)
(v: Double) => List(v)
(acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2

(Norte,1200.0)
(Sur,45.0)
(Norte,350.0)
(Sur,1200.0)
(Norte,25.0)
(Sur,350.0)
(Norte,45.0)
Sur → List(45.0, 1200.0, 350.0)
Norte → List(1200.0, 350.0, 25.0, 45.0)


import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@71ca91b3
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@3bc23217
ventas: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[7] at parallelize at cmd4.sc:12
ventasPorRegion: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[8] at map at cmd4.sc:23
importesPorRegion: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[9] at combineByKey at cmd4.sc:43
res4_9: Double => List[Double] = ammonite.$sess.cmd4$Helper$$Lambda$6286/0x000001c8823ccf00@2ce28f53
res4_10: (List[Double], List[Double]) => List[Double] = ammonite.$sess.cmd4$Helper$$Lambda$6287/0x000001c8823cd2c0@26cd75a8

In [2]:
val ventasPorRegion = ventas.map { linea =>
  val campos = linea.split(",")
  val region = campos(1)
  val importe = campos(2).toDouble
  (region, importe)
}

ventasPorRegion: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[3] at map at cmd2.sc:1

In [5]:
val importesPorRegion = ventasPorRegion.combineByKey(
  (v: Double) => List(v),
  (acc: List[Double], v: Double) => acc :+ v,
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2
)

importesPorRegion: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[10] at combineByKey at cmd5.sc:4

In [6]:
val importesPorRegion = ventasPorRegion.combineByKey(

  (v: Double) => {
    println(s"[CREATE] Nuevo acumulador con: $v")
    List(v)
  },

  (acc: List[Double], v: Double) => {
    println(s"[MERGE VALUE] Lista actual: $acc | Añadiendo: $v")
    acc :+ v
  },

  (acc1: List[Double], acc2: List[Double]) => {
    println(s"[MERGE COMBINERS] Uniendo: $acc1 y $acc2")
    acc1 ++ acc2
  }

)

importesPorRegion.collect().foreach(println)

[CREATE] Nuevo acumulador con: 1200.0
[CREATE] Nuevo acumulador con: 25.0
[CREATE] Nuevo acumulador con: 45.0
[CREATE] Nuevo acumulador con: 350.0
[CREATE] Nuevo acumulador con: 45.0
[CREATE] Nuevo acumulador con: 350.0
[CREATE] Nuevo acumulador con: 1200.0
[MERGE COMBINERS] Uniendo: List(45.0) y List(1200.0)
[MERGE COMBINERS] Uniendo: List(1200.0) y List(350.0)
[MERGE COMBINERS] Uniendo: List(45.0, 1200.0) y List(350.0)
[MERGE COMBINERS] Uniendo: List(1200.0, 350.0) y List(25.0)
[MERGE COMBINERS] Uniendo: List(1200.0, 350.0, 25.0) y List(45.0)
(Sur,List(45.0, 1200.0, 350.0))
(Norte,List(1200.0, 350.0, 25.0, 45.0))


importesPorRegion: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[11] at combineByKey at cmd6.sc:13

In [7]:
def verParticiones[T](rdd: org.apache.spark.rdd.RDD[T]): Unit = {
  rdd.mapPartitionsWithIndex { (index, iter) =>
    Iterator(s"📦 Partición $index -> ${iter.toList.mkString(", ")}")
  }.collect().foreach(println)
}

defined function verParticiones

In [8]:
val notas = sc.parallelize(List(("Carlos", 8.5), ("Marta", 7.0), ("Carlos", 9.0), ("Ana", 6.5), ("Marta", 8.0), ("Ana", 7.5), ("Luis", 5.5), ("Carlos", 10.0), ("Luis", 6.0), ("Marta", 9.0), ("Ana", 8.5), ("Luis", 7.0)))

val evaluacionPorAlumno = notas.combineByKey(
  (v: Double) => List(v),
  (acc: List[Double], v: Double) => acc :+ v,
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2
)

evaluacionPorAlumno.collect().foreach { case (alumno, lista) =>
  println(s"$alumno - $lista")
}




2026-04-27T07:59:42.428813600Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

notas: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[12] at parallelize at cmd8.sc:1
evaluacionPorAlumno: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[13] at combineByKey at cmd8.sc:6

In [9]:
val temperaturas = sc.parallelize(List( ("Madrid", 18.5),("Barcelona", 20.0), ("Madrid", 21.0), ("Valencia", 22.5), ("Sevilla", 25.0),("Barcelona", 19.5), ("Madrid", 17.0), ("Valencia", 23.0), ("Sevilla", 26.5), ("Barcelona", 18.0),("Madrid", 20.5), ("Valencia", 21.5)
), 3)

verParticiones(temperaturas)
 
 val temperaturasPorCiudad = temperaturas.combineByKey(
    (v:Double)  => List(v),
    (acc:List [Double] , v:Double) => acc :+ v,
    (acc1: List[Double], acc2:List[Double])=> acc1 ++ acc2
    
 )

 temperaturasPorCiudad.collect().foreach { case (temperatura,lista) =>
   println(s"$temperatura - $lista ")
   }

2026-04-27T08:19:06.758873900Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

temperaturas: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[14] at parallelize at cmd9.sc:1
temperaturasPorCiudad: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[16] at combineByKey at cmd9.sc:9

In [10]:
val compras = sc.parallelize(List( ("Ana", "Laptop"), ("Luis", "Teclado"), ("Ana", "Mouse"),  ("Marta", "Monitor"),("Luis", "Tablet"), ("Ana", "Auriculares"), ("Carlos", "Impresora"),("Marta", "Webcam"), ("Luis", "Ratón"), ("Carlos", "SSD"), ("Ana", "USB"), ("Marta", "Silla")
), 3)

verParticiones(compras)

val totalDeCompras = compras.combineByKey(
  (v: String) => List(v),
  (acc: List[String], v: String) => acc :+ v,
  (acc1: List[String], acc2: List[String]) => acc1 ++ acc2
)

totalDeCompras.collect().foreach {case  (cliente,lista) =>
  println(s"$cliente - $lista")}

2026-04-27T08:27:05.282029Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.cor

compras: org.apache.spark.rdd.RDD[(String, String)] = ParallelCollectionRDD[17] at parallelize at cmd10.sc:1
totalDeCompras: org.apache.spark.rdd.RDD[(String, List[String])] = ShuffledRDD[19] at combineByKey at cmd10.sc:9

In [11]:
val palabras = sc.parallelize(List( ("A", "Azure"),("S", "Spark"), ("A", "Almond"), ("S", "Scala"), ("P", "Python"), ("J", "Java"),  ("A", "Airflow"), ("S", "SQL"), ("P", "PostgreSQL"), ("J", "Jupyter"), ("A", "AWS"),("P", "PySpark")
), 3)

verParticiones(palabras)
val totalDeLetras = palabras.combineByKey(
  (v: String) => List(v),
  (acc: List[String], v: String) => acc :+ v,
  (acc1: List[String], acc2: List[String]) => acc1 ++ acc2
)

// 2. Corregido: totalDePalabras SIN paréntesis y CON .collect()
totalDeLetras.collect().foreach { case (objetos, lista) =>
  println(s"$objetos - $lista")
}


2026-04-27T08:33:53.046304300Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

palabras: org.apache.spark.rdd.RDD[(String, String)] = ParallelCollectionRDD[20] at parallelize at cmd11.sc:1
totalDeLetras: org.apache.spark.rdd.RDD[(String, List[String])] = ShuffledRDD[22] at combineByKey at cmd11.sc:8

In [25]:
val eventos = sc.parallelize(List(
  ("user1", "video", 10.0),
  ("user2", "quiz", 5.0),
  ("user1", "video", 15.0),
  ("user3", "video", 20.0),
  ("user2", "video", 8.0),
  ("user1", "quiz", 7.0),
  ("user3", "quiz", 6.0),
  ("user2", "video", 12.0),
  ("user1", "video", 9.0),
  ("user3", "video", 11.0),
  ("user2", "quiz", 4.0),
  ("user3", "video", 13.0)
), 3)
verParticiones(eventos)


val eventosPorUsuario = eventos.map(t => (t._1, t._3))


val resultado = eventosPorUsuario.combineByKey(
  (v: Double) => List(v),
  (acc: List[Double], v: Double) => acc :+ v,
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2
)


resultado.collect().foreach { case (usuario, lista) =>
  println(s"$usuario - $lista")
}



📦 Partición 0 -> (user1,video,10.0), (user2,quiz,5.0), (user1,video,15.0), (user3,video,20.0)
📦 Partición 1 -> (user2,video,8.0), (user1,quiz,7.0), (user3,quiz,6.0), (user2,video,12.0)
📦 Partición 2 -> (user1,video,9.0), (user3,video,11.0), (user2,quiz,4.0), (user3,video,13.0)
user3 - List(20.0, 6.0, 11.0, 13.0)
user1 - List(10.0, 15.0, 7.0, 9.0)
user2 - List(5.0, 8.0, 12.0, 4.0)


eventos: org.apache.spark.rdd.RDD[(String, String, Double)] = ParallelCollectionRDD[67] at parallelize at cmd25.sc:1
eventosPorUsuario: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[69] at map at cmd25.sc:18
resultado: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[70] at combineByKey at cmd25.sc:24

In [26]:
val acumulado = eventosPorUsuario.combineByKey(
  (v: Double) => (v, 1),                          
  (acc: (Double, Int), v: Double) => (acc._1 + v, acc._2 + 1),
  (acc1: (Double, Int), acc2: (Double, Int)) => (acc1._1 + acc2._1, acc1._2 + acc2._2) 
)


acumulado.collect().foreach { case (usuario, (suma, totalEventos)) =>
  val promedio = suma / totalEventos
  println(s"Usuario: $usuario | Suma: $suma | Promedio: $promedio")
}

Usuario: user3 | Suma: 50.0 | Promedio: 12.5
Usuario: user1 | Suma: 41.0 | Promedio: 10.25
Usuario: user2 | Suma: 29.0 | Promedio: 7.25


acumulado: org.apache.spark.rdd.RDD[(String, (Double, Int))] = ShuffledRDD[71] at combineByKey at cmd26.sc:4

In [ ]:
En cada partición ocurren los acciones principales :
    CreateCombines en la cual se ve una clave en esa particion y se usa para inicicar el acumulador .
    Megavalue : si vuelve a aparecer la partición usa esa función para añadir el nuevo acmumulador que ya existía.

    Se ejecuta en la fase final .Se necesita jumtar los resultados en todas las particiones que viven en distintos nodos . MergeCombiners toma los acumuladores t los fusiona en uno solo por cada clave.!=

    El conmbineByKey es mas flexible ya que permite campbiar el tipo de dato de salida ya sea meter una Tupla, sacar un LIst o un Double .En cambio reduceKey el resultado tiene que ser de la misma entrada.!=
    El uso de reduceKEy es malo ya que mueve los datos en registros individuales y tiene el riesgo de que la memoria de dicho cluster se colpase con tanta informacíón.

In [27]:
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("DataFrames_Dia14")   // nombre visible en la Spark UI
  .master("local[*]")            // modo local, todos los cores disponibles
  .getOrCreate()                 // crea o reutiliza una sesión existente

// Desde SparkSession podemos acceder al SparkContext si lo necesitamos
val sc = spark.sparkContext

println(s"Spark ${spark.version} listo")

2026-04-27T09:10:13.137831700Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@71ca91b3
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@3bc23217

In [28]:
// 1. Importar los implicits de la sesión (asumiendo que tu sesión se llama 'spark')
import spark.implicits._

// 2. Ahora ya puedes usar .toDF()
val dfEmpleados = Seq(
  (1, "Ana García", 28, "Ingeniería"),
  (2, "Luis Martínez", 34, "Marketing"),
  (3, "Marta López", 22, "Ingeniería"),
  (4, "Pedro Ruiz", 41, "Dirección")
).toDF("id", "nombre", "edad", "departamento")

dfEmpleados.show()


+---+-------------+----+------------+
| id|       nombre|edad|departamento|
+---+-------------+----+------------+
|  1|   Ana García|  28|  Ingeniería|
|  2|Luis Martínez|  34|   Marketing|
|  3|  Marta López|  22|  Ingeniería|
|  4|   Pedro Ruiz|  41|   Dirección|
+---+-------------+----+------------+



import spark.implicits._
dfEmpleados: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 2 more fields]

In [32]:

import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val ruta = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
Files.createDirectories(Paths.get(ruta))

val contenidoCSV =
"""id_venta,fecha,producto,categoria,cantidad,precio_unitario,ciudad
1,2026-04-01,Portátil,Informática,2,850.50,Madrid
2,2026-04-01,Ratón,Informática,5,18.90,Valencia
3,2026-04-02,Teclado,Informática,3,45.00,Sevilla
4,2026-04-02,Monitor,Informática,1,199.99,Madrid
5,2026-04-03,Silla,Oficina,4,120.00,Barcelona
6,2026-04-03,Mesa,Oficina,2,250.00,Zaragoza
7,2026-04-04,Webcam,Informática,6,39.90,Madrid
8,2026-04-04,Auriculares,Informática,3,59.99,Valencia
"""

val pathCSV = s"$ruta/ventas.csv"

Files.write(
  Paths.get(pathCSV),
  contenidoCSV.getBytes(StandardCharsets.UTF_8)
)

println(s"CSV creado en: $pathCSV")


CSV creado en: C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/ventas.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
res32_3: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos
contenidoCSV: String = """id_venta,fecha,producto,categoria,cantidad,precio_unitario,ciudad
1,2026-04-01,Portátil,Informática,2,850.50,Madrid
2,2026-04-01,Ratón,Informática,5,18.90,Valencia
3,2026-04-02,Teclado,Informática,3,45.00,Sevilla
4,2026-04-02,Monitor,Informática,1,199.99,Madrid
5,2026-04-03,Silla,Oficina,4,120.00,Barcelona
6,2026-04-03,Mesa,Oficina,2,250.00,Zaragoza
7,2026-04-04,Webcam,Informática,6,39.90,Madrid
8,2026-04-04,Auriculares,Informática,3,59.99,Valencia
"""
pathCSV: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/ventas.csv"
res32_6: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos\ventas.csv

In [33]:
val dfVentas = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/ventas.csv")

dfVentas.show(5)
dfVentas.printSchema()

+--------+----------+--------+-----------+--------+---------------+---------+
|id_venta|     fecha|producto|  categoria|cantidad|precio_unitario|   ciudad|
+--------+----------+--------+-----------+--------+---------------+---------+
|       1|2026-04-01|Portátil|Informática|       2|          850.5|   Madrid|
|       2|2026-04-01|   Ratón|Informática|       5|           18.9| Valencia|
|       3|2026-04-02| Teclado|Informática|       3|           45.0|  Sevilla|
|       4|2026-04-02| Monitor|Informática|       1|         199.99|   Madrid|
|       5|2026-04-03|   Silla|    Oficina|       4|          120.0|Barcelona|
+--------+----------+--------+-----------+--------+---------------+---------+
only showing top 5 rows
root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- ciudad: string (nul

dfVentas: org.apache.spark.sql.package.DataFrame = [id_venta: int, fecha: date ... 5 more fields]

In [35]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val ruta = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos" // Usa tu propia ruta, sino te dará error
Files.createDirectories(Paths.get(ruta))

val jsonSimple =
"""{"id":1,"nombre":"Ana García","edad":28,"departamento":"Ingeniería"}
{"id":2,"nombre":"Luis Martínez","edad":34,"departamento":"Marketing"}
{"id":3,"nombre":"Marta López","edad":22,"departamento":"Ingeniería"}
{"id":4,"nombre":"Pedro Ruiz","edad":41,"departamento":"Dirección"}"""

val rutaJsonSimple = s"$ruta/empleados_simple.json"

Files.write(
  Paths.get(rutaJsonSimple),
  jsonSimple.getBytes(StandardCharsets.UTF_8)
)

println(s"JSON simple creado en: $rutaJsonSimple")

JSON simple creado en: C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_simple.json


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
res35_3: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos
jsonSimple: String = """{"id":1,"nombre":"Ana García","edad":28,"departamento":"Ingeniería"}
{"id":2,"nombre":"Luis Martínez","edad":34,"departamento":"Marketing"}
{"id":3,"nombre":"Marta López","edad":22,"departamento":"Ingeniería"}
{"id":4,"nombre":"Pedro Ruiz","edad":41,"departamento":"Dirección"}"""
rutaJsonSimple: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_simple.json"
res35_6: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos\empleados_simple.json

In [36]:
val dfJsonSimple = spark.read
  .option("inferSchema", "true")
  .json(rutaJsonSimple)

dfJsonSimple.show(false)
dfJsonSimple.printSchema()

+------------+----+---+-------------+
|departamento|edad|id |nombre       |
+------------+----+---+-------------+
|Ingeniería  |28  |1  |Ana García   |
|Marketing   |34  |2  |Luis Martínez|
|Ingeniería  |22  |3  |Marta López  |
|Dirección   |41  |4  |Pedro Ruiz   |
+------------+----+---+-------------+

root
 |-- departamento: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)



dfJsonSimple: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

In [37]:
val jsonMultilinea =
"""[
  {
    "id": 1,
    "nombre": "Ana García",
    "edad": 28,
    "departamento": "Ingeniería"
  },
  {
    "id": 2,
    "nombre": "Luis Martínez",
    "edad": 34,
    "departamento": "Marketing"
  },
  {
    "id": 3,
    "nombre": "Marta López",
    "edad": 22,
    "departamento": "Ingeniería"
  },
  {
    "id": 4,
    "nombre": "Pedro Ruiz",
    "edad": 41,
    "departamento": "Dirección"
  }
]"""

val rutaJsonMultilinea = s"$ruta/empleados_multilinea.json"

Files.write(
  Paths.get(rutaJsonMultilinea),
  jsonMultilinea.getBytes(StandardCharsets.UTF_8)
)

println(s"JSON multilínea creado en: $rutaJsonMultilinea")

JSON multilínea creado en: C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_multilinea.json


jsonMultilinea: String = """[
  {
    "id": 1,
    "nombre": "Ana García",
    "edad": 28,
    "departamento": "Ingeniería"
  },
  {
    "id": 2,
    "nombre": "Luis Martínez",
    "edad": 34,
    "departamento": "Marketing"
  },
  {
    "id": 3,
    "nombre": "Marta López",
    "edad": 22,
    "departamento": "Ingeniería"
  },
  {
    "id": 4,
    "nombre": "Pedro Ruiz",
    "edad": 41,
    "departamento": "Dirección"
  }
]"""
rutaJsonMultilinea: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_multilinea.json"
res37_2: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos\empleados_multilinea.json

In [38]:
val dfJsonMultilinea = spark.read
  .option("multiline", "true")
  .option("inferSchema", "true")
  .json(rutaJsonMultilinea)

dfJsonMultilinea.show(false)
dfJsonMultilinea.printSchema()

+------------+----+---+-------------+
|departamento|edad|id |nombre       |
+------------+----+---+-------------+
|Ingeniería  |28  |1  |Ana García   |
|Marketing   |34  |2  |Luis Martínez|
|Ingeniería  |22  |3  |Marta López  |
|Dirección   |41  |4  |Pedro Ruiz   |
+------------+----+---+-------------+

root
 |-- departamento: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)



dfJsonMultilinea: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

In [39]:
val dfVentas = spark.read
  .option("header", "true")       // primera fila como nombres de columna
  .option("inferSchema", "true")  // detectar tipos automáticamente
  .csv("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/ventas.csv")

dfVentas.show(5)         // muestra las primeras 5 filas


+--------+----------+--------+-----------+--------+---------------+---------+
|id_venta|     fecha|producto|  categoria|cantidad|precio_unitario|   ciudad|
+--------+----------+--------+-----------+--------+---------------+---------+
|       1|2026-04-01|Portátil|Informática|       2|          850.5|   Madrid|
|       2|2026-04-01|   Ratón|Informática|       5|           18.9| Valencia|
|       3|2026-04-02| Teclado|Informática|       3|           45.0|  Sevilla|
|       4|2026-04-02| Monitor|Informática|       1|         199.99|   Madrid|
|       5|2026-04-03|   Silla|    Oficina|       4|          120.0|Barcelona|
+--------+----------+--------+-----------+--------+---------------+---------+
only showing top 5 rows


dfVentas: org.apache.spark.sql.package.DataFrame = [id_venta: int, fecha: date ... 5 more fields]

In [40]:
val dfClientes = spark.read
  .option("multiline", "true")    // para JSON con objetos multilínea
  .json("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_multilinea.json")

dfClientes.show()

2026-04-27T09:45:53.812172600Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

dfClientes: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

In [41]:
val dfClientes = spark.read
  .option("multiline", "true")    // para JSON con objetos multilínea
  .json("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_multilinea.json")

dfClientes.show()

+------------+----+---+-------------+
|departamento|edad| id|       nombre|
+------------+----+---+-------------+
|  Ingeniería|  28|  1|   Ana García|
|   Marketing|  34|  2|Luis Martínez|
|  Ingeniería|  22|  3|  Marta López|
|   Dirección|  41|  4|   Pedro Ruiz|
+------------+----+---+-------------+



dfClientes: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

In [42]:
dfEmpleados.printSchema()


root
 |-- id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = false)
 |-- departamento: string (nullable = true)



In [43]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

import org.apache.spark.sql.types._

val ruta = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
Files.createDirectories(Paths.get(ruta))

val contenidoPedidos =
"""id_pedido,id_cliente,fecha,importe,completado
1,101,2026-04-01,250.75,true
2,102,2026-04-01,99.90,false
3,103,2026-04-02,430.00,true
4,101,2026-04-03,120.50,true
5,104,2026-04-04,75.25,false
6,105,2026-04-05,680.40,true
"""

val rutaPedidos = s"$ruta/pedidos.csv"

Files.write(
  Paths.get(rutaPedidos),
  contenidoPedidos.getBytes(StandardCharsets.UTF_8)
)

println(s"CSV creado correctamente en: $rutaPedidos")

CSV creado correctamente en: C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/pedidos.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import org.apache.spark.sql.types._
ruta: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
res43_4: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos
contenidoPedidos: String = """id_pedido,id_cliente,fecha,importe,completado
1,101,2026-04-01,250.75,true
2,102,2026-04-01,99.90,false
3,103,2026-04-02,430.00,true
4,101,2026-04-03,120.50,true
5,104,2026-04-04,75.25,false
6,105,2026-04-05,680.40,true
"""
rutaPedidos: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/pedidos.csv"
res43_7: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos\pedidos.csv

In [45]:
import org.apache.spark.sql.types._
val schemaPedidos = StructType(List(
  StructField("id_pedido", IntegerType, nullable = false),
  StructField("id_cliente", IntegerType, nullable = false),
  StructField("fecha", StringType, nullable = true),
  StructField("importe", DoubleType, nullable = true),
  StructField("completado", BooleanType, nullable = true)
))

val dfPedidos = spark.read
  .option("header", "true")
  .schema(schemaPedidos)
  .csv("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/pedidos.csv")

dfPedidos.show(false)
dfPedidos.printSchema()

+---------+----------+----------+-------+----------+
|id_pedido|id_cliente|fecha     |importe|completado|
+---------+----------+----------+-------+----------+
|1        |101       |2026-04-01|250.75 |true      |
|2        |102       |2026-04-01|99.9   |false     |
|3        |103       |2026-04-02|430.0  |true      |
|4        |101       |2026-04-03|120.5  |true      |
|5        |104       |2026-04-04|75.25  |false     |
|6        |105       |2026-04-05|680.4  |true      |
+---------+----------+----------+-------+----------+

root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: string (nullable = true)
 |-- importe: double (nullable = true)
 |-- completado: boolean (nullable = true)



import org.apache.spark.sql.types._
schemaPedidos: StructType = Seq(
  StructField(
    name = "id_pedido",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "id_cliente",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "fecha",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "importe",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "completado",
    dataType = BooleanType,
    nullable = true,
    metadata = {}
  )
)
dfPedidos: org.apache.spark.sql.package.DataFrame = [id_pedido: int, id_cliente: int ... 3 more fields]

In [49]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._

val spark = SparkSession.builder()
  .appName("Dia14S1_DataFrames")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

import spark.implicits._   // necesario para .toDF() y otras conversiones

spark.sparkContext.setLogLevel("ERROR")

println(s"✅ Spark ${spark.version} listo")


2026-04-27T09:52:00.795453400Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@71ca91b3
import spark.implicits._

In [50]:
// Crear DataFrame desde una secuencia de tuplas
val dfCursos = Seq(
  (1, "Big Data con Scala", "Avanzado",  140, 4.8),
  (2, "Python para Datos",  "Intermedio", 80, 4.6),
  (3, "SQL Empresarial",    "Básico",     40, 4.9),
  (4, "Machine Learning",   "Avanzado",  120, 4.7),
  (5, "Power BI",           "Básico",     60, 4.5)
).toDF("id", "titulo", "nivel", "horas", "valoracion")

// Ver las filas
println("=== Contenido del DataFrame ===")
dfCursos.show()

// Ver la estructura
println("=== Schema del DataFrame ===")
dfCursos.printSchema()

// Número de filas y columnas
println(s"Filas: ${dfCursos.count()}")
println(s"Columnas: ${dfCursos.columns.length}")
println(s"Nombres de columnas: ${dfCursos.columns.mkString(", ")}")

=== Contenido del DataFrame ===
+---+------------------+----------+-----+----------+
| id|            titulo|     nivel|horas|valoracion|
+---+------------------+----------+-----+----------+
|  1|Big Data con Scala|  Avanzado|  140|       4.8|
|  2| Python para Datos|Intermedio|   80|       4.6|
|  3|   SQL Empresarial|    Básico|   40|       4.9|
|  4|  Machine Learning|  Avanzado|  120|       4.7|
|  5|          Power BI|    Básico|   60|       4.5|
+---+------------------+----------+-----+----------+

=== Schema del DataFrame ===
root
 |-- id: integer (nullable = false)
 |-- titulo: string (nullable = true)
 |-- nivel: string (nullable = true)
 |-- horas: integer (nullable = false)
 |-- valoracion: double (nullable = false)

Filas: 5
Columnas: 5
Nombres de columnas: id, titulo, nivel, horas, valoracion


dfCursos: org.apache.spark.sql.package.DataFrame = [id: int, titulo: string ... 3 more fields]

In [51]:
// Estadísticas descriptivas de columnas numéricas
println("=== Estadísticas descriptivas ===")
dfCursos.describe("horas", "valoracion").show()

=== Estadísticas descriptivas ===
+-------+-----------------+------------------+
|summary|            horas|        valoracion|
+-------+-----------------+------------------+
|  count|                5|                 5|
|   mean|             88.0|               4.7|
| stddev|41.47288270665544|0.1581138830084192|
|    min|               40|               4.5|
|    max|              140|               4.9|
+-------+-----------------+------------------+



In [60]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._

val spark = SparkSession.builder()
  .appName("Dia14S1_DataFrames")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

import spark.implicits._   // necesario para .toDF() y otras conversiones

spark.sparkContext.setLogLevel("ERROR")

println(s"✅ Spark ${spark.version} listo")

✅ Spark 4.1.1 listo


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.types._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@71ca91b3
import spark.implicits._